# Model Training - Terrain Vente Marrakech (Version 46% R2)
Ce notebook reproduit exactement la configuration ayant obtenu un score R2 de 0.46.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score

# Ajouter le chemin vers la racine pour importer les helpers
sys.path.append('../')
from model_training_helpers import split_data, model_pipeline, metric_model, get_features, print_metrics, save_model, preprocess_data

print('Modules chargés avec succès.')

## 1. Chargement des données

In [ ]:
csv_path = '../data/cleaned_data/vente/terrain_vente_cleaned.csv'
df = pd.read_csv(csv_path)
print(f'Données chargées : {df.shape}')

## 2. Préparation et Agence Tier

In [ ]:
df = preprocess_data(df, property_type='terrain')
X_train, X_test, y_train, y_test = split_data(df, target='prix_num')

# Calcul de l'Agence Tier (Train only pour éviter le leakage)
agency_m2 = (y_train / X_train['surface_num']).groupby(X_train['agence']).median()
X_train['agence_tier'] = X_train['agence'].map(agency_m2).fillna(agency_m2.median())
X_test['agence_tier'] = X_test['agence'].map(agency_m2).fillna(agency_m2.median())

num_features, cat_features = get_features(X_train)
print(f'Features prêtes : {X_train.shape[1]}')

## 3. Entraînement (Configuration 46% R2)

In [ ]:
print("Entraînement du modèle gagnant...")

# Configuration exacte de test_honest.py
model = xgb.XGBRegressor(
    n_estimators=1200, 
    learning_rate=0.02, 
    max_depth=6, 
    random_state=42,
    n_jobs=-1
)

pipeline = model_pipeline(num_features, cat_features, model, all_columns=X_train.columns.tolist())

# Pondération linéaire par le prix
weights = y_train / y_train.mean()

pipeline.fit(X_train, np.log1p(y_train), model__sample_weight=weights)
print('Entraînement terminé.')

## 4. Évaluation

In [ ]:
y_pred = np.expm1(pipeline.predict(X_test))
metrics = metric_model(y_test, y_pred, model_name='Terrain 46% R2')
print_metrics(metrics, model_name='Terrain 46% R2')

## 5. Sauvegarde

In [ ]:
save_model(pipeline, 'terrain_vente_46pct_model.joblib')
print('Modèle sauvegardé.')